--------------------------
#### Semantic text search using embeddings
------------------------------
- Semantic Search Efficiency:
    - Utilizing embeddings allows for efficient semantic search through all reviews.
    - The process involves embedding the search query and then identifying the most similar reviews.
    - This method enables quick retrieval of relevant reviews based on semantic similarity.
    
- Low Cost:
    - The cost-effectiveness of the search process is emphasized.
    - Embedding-based search minimizes computational expenses while maintaining search accuracy.
    - This approach offers an economical solution for exploring reviews in a semantically meaningful way.

In [1]:
import pandas as pd
import numpy as np

from ast import literal_eval

#### load the saved embeddings (food reviews)

In [3]:
datafile_path = r"D:\AI-DATASETS\02-MISC-large\GenAI-LLMs\amazon_food_reviews_with_embeddings_2k.csv"

In [4]:
df = pd.read_csv(datafile_path)
df.shape

(2000, 9)

In [5]:
df.sample(3)

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding
473,200017,B009NEQAHQ,A1CN3VUX1DQ0FN,5,"Great taste, beautiful crystal!",My wife and I use this everyday in our dinners...,"Title: Great taste, beautiful crystal!; Conten...",108,"[0.013699831441044807, 0.026147527620196342, -..."
1777,93366,B007TGDXMU,AAMUNRK134Y5P,5,Very pleased,"Good price, good quality with convenience. Fu...","Title: Very pleased; Content: Good price, good...",39,"[0.005218738690018654, -0.04089787229895592, -..."
204,41459,B0070I4OI4,A337QOP1S8B583,1,poor,candy was old. Paper stuck to the candies. Als...,Title: poor; Content: candy was old. Paper stu...,35,"[-0.02971717156469822, -0.006954488344490528, ..."


In [6]:
df.dtypes

Unnamed: 0        int64
ProductId        object
UserId           object
Score             int64
Summary          object
Text             object
combined         object
n_tokens          int64
ada_embedding    object
dtype: object

In [7]:
%%time
# convert string to array
df["embedding"] = df.ada_embedding.apply(literal_eval).apply(np.array)

CPU times: total: 12.4 s
Wall time: 13.3 s


#### Search query

- compare the cosine similarity of the embeddings of the query and the documents, and show top_n best matches.

In [8]:
from openai import OpenAI
import json
import os

In [9]:
openai_api_key = os.environ.get('OPENAI_API_KEY')

In [20]:
client = OpenAI()

In [21]:
# models
EMBEDDING_MODEL = "text-embedding-3-small"
GPT_MODEL       = "gpt-3.5-turbo"

In [22]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_cosine_similarity(document_embeddings, query_embedding):
    """
    Calculate cosine similarity between the query embedding and document embeddings.

    Parameters:
    - query_embedding: The embedding vector for the search query.
    - document_embeddings: List of embeddings for the documents.

    Returns:
    - List of cosine similarities between the query and each document.
    """
    similarities = cosine_similarity(query_embedding, document_embeddings)
    return similarities.flatten()

In [23]:
# For a specific query and documents
query_embedding     = np.array([[0.1, 0.3, 0.5]])  

document_embeddings = np.array([
    [0.2, 0.4, 0.6],  
    [0.15, 0.35, 0.55],
   
])

In [24]:
query_embedding.reshape(-1, 1)

array([[0.1],
       [0.3],
       [0.5]])

In [25]:
get_cosine_similarity(query_embedding, document_embeddings)

array([0.99385869, 0.99808276])

In [26]:
# search through the reviews for a specific product
def get_sim_scores(df, query):
    
    response = client.embeddings.create(
                    model          = EMBEDDING_MODEL, 
                    input          = query, 
                    encoding_format= "float"
    )
    
    # obtain the embedding for the query
    query_embedding = np.array(response.data[0].embedding)[np.newaxis, :]
       
    df["similarity"] = df.embedding.apply(lambda x: get_cosine_similarity(x[np.newaxis, :], query_embedding))
    
    return df

In [27]:
df_with_sim_scores = get_sim_scores(df, "delicious beans")

In [28]:
df_with_sim_scores

,Unnamed: 0,ProductId,UserId,Score,Summary,Text,combined,n_tokens,ada_embedding,embedding,similarity
0,467784,B000V1O28Y,A1CHFK7Z8ZJBQ7,4,Better than I thought they would be!,These were actually quite tasty despite the ex...,Title: Better than I thought they would be!; C...,58,"[0.01103049237281084, -0.04996507987380028, -0...","[0.01103049237281084, -0.04996507987380028, -0...",[0.36272377530521527]
1,102286,B001QEB3E6,AY2O7I9BNG2UI,4,"Dog loves them, but I hate the smell","I have a 70 pound lab mix, and he goes nuts fo...","Title: Dog loves them, but I hate the smell; C...",98,"[0.008594190701842308, 0.0012299544177949429, ...","[0.008594190701842308, 0.0012299544177949429, ...",[0.273066745643227]
2,228586,B00295IGHS,A37VSXI1MDHBWJ,2,Too runny for my taste,"I bought this to use to give my dogs pills, so...",Title: Too runny for my taste; Content: I boug...,100,"[0.020592087879776955, 0.016811959445476532, -...","[0.020592087879776955, 0.016811959445476532, -...",[0.24736987318507253]
3,149371,B006BXUKAA,A2Y2RTN8AZ9C7F,2,Smells like the beans are flavored with alcohol,"When I opened the back, the overwhelming alcoh...",Title: Smells like the beans are flavored with...,88,"[-0.03934091702103615, -0.0028343461453914642,...","[-0.03934091702103615, -0.0028343461453914642,...",[0.40993948932098323]
4,175998,B007TGO1U8,A2FHXIKEYLW2C0,1,Aftertaste,"I am disappointed in this product. First, ther...",Title: Aftertaste; Content: I am disappointed ...,79,"[-0.002285151509568095, -0.01151236705482006, ...","[-0.002285151509568095, -0.01151236705482006, ...",[0.216486180924231]
...,...,...,...,...,...,...,...,...,...,...,...
1995,7177,B004OQLIHK,AKHQMSUORSA91,5,Delicious!,I have ordered these raisins multiple times. ...,Title: Delicious!; Content: I have ordered the...,43,"[0.015759458765387535, -0.039189912378787994, ...","[0.015759458765387535, -0.039189912378787994, ...",[0.4325459919791673]
1996,401971,B0006349W6,A21BT40VZCCYT4,5,Good Training Treat,My dog will come in from outside when I am tra...,Title: Good Training Treat; Content: My dog wi...,48,"[-0.023052580654621124, -0.013884083367884159,...","[-0.023052580654621124, -0.013884083367884159,...",[0.177988404183951]
1997,462087,B00611F084,A6D4ND3C3BCYV,5,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...,Title: Jamica Me Crazy Coffee; Content: Wolfga...,40,"[-0.029660701751708984, -0.045804813504219055,...","[-0.029660701751708984, -0.045804813504219055,...",[0.28664891220785516]
1998,267548,B005QKH5HA,A3LR9HCV3D96I3,5,Party Peanuts,Great product for the price. Mix with the Asia...,Title: Party Peanuts; Content: Great product f...,45,"[0.0010645465226843953, -0.022560352459549904,...","[0.0010645465226843953, -0.022560352459549904,...",[0.34875262331316415]


In [29]:
df_with_sim_scores_sorted = df_with_sim_scores.sort_values("similarity", ascending=False)

In [30]:
df_with_sim_scores_sorted.columns

Index(['Unnamed: 0', 'ProductId', 'UserId', 'Score', 'Summary', 'Text',
       'combined', 'n_tokens', 'ada_embedding', 'embedding', 'similarity'],
      dtype='object')

In [31]:
pd.set_option('max_colwidth', 300)

In [33]:
# Selecting specific columns (e.g., 'column1', 'column2') from the sorted DataFrame
df_with_sim_scores_sorted[['combined', 'similarity']].sample(3)

,combined,similarity
499,Title: Makes the BEST Carne Asada; Content: Put this marinade over 2 lbs Beef Skirt Steak (Arrachera) with sliced onions overnight and then bbq over high heat-OMG delish!,[0.3491646838368057]
1230,Title: I couldn't love the double bergamot more; Content: I'm absolutely addicted to this tea. it's the best tea I've ever had. when Starbucks changed their black iced tea I had to find a replacement. this stuff is better than anything ever. I'm very passionate about tea. I've never had it hot b...,[0.28997895884844804]
1057,Title: expectations met; Content: Is what it says it is would never think it came out of a can would not be suprised to see this product on major. store shelves on account of quality,[0.2647110335066889]


In [35]:
# search through the reviews for a specific product
def search_reviews(df, query, n=5):
    
    response = client.embeddings.create(
                    model          = EMBEDDING_MODEL, 
                    input          = query, 
                    encoding_format= "float"
    )
    
    # obtain the embedding for the query
    query_embedding = np.array(response.data[0].embedding)[np.newaxis, :]
       
    df["similarity"] = df.embedding.apply(lambda x: get_cosine_similarity(x[np.newaxis, :], query_embedding))
    
    df_with_sim_scores_sorted = df_with_sim_scores.sort_values("similarity", ascending=False)
    
    # Selecting specific columns (e.g., 'column1', 'column2') from the sorted DataFrame
    results_df = df_with_sim_scores_sorted[['combined', 'similarity']].head(n)
    
    return results_df

In [36]:
search_reviews(df, "delicious beans", n=3)

,combined,similarity
73,"Title: Rancho Gordo Beans - What fun!; Content: I've probably tried about 15 varieties of beans from Rancho Gordo. They're all excellent. Some are better suited for one type dish than another, and if you visit their website, they will tell you what works best for which dish.<br /><br />Some ar...",[0.613006029797154]
1088,"Title: Delicious!; Content: I enjoy this white beans seasoning, it gives a rich flavor to the beans I just love it, my mother in law didn't know about this Zatarain's brand and now she is traying different seasoning and she likes it very much.<br />Thank you Amazon for having it because now I ca...",[0.5735619215044223]
1927,Title: Fantastic Instant Refried beans; Content: Fantastic Instant Refried Beans have been a staple for my family now for nearly 20 years. All 7 of us love it and my grown kids are passing on the tradition.,[0.5581251778459062]


In [37]:
search_reviews(df, "whole wheat pasta", n=3)

,combined,similarity
1038,"Title: Tasty and Quick Pasta; Content: Barilla Whole Grain Fusilli with Vegetable Marinara is tasty and has an excellent chunky vegetable marinara. I just wish there was more of it. If you aren't starving or on a diet, the 9oz serving is enough for lunch although you might want to add a piece ...",[0.4915431869235446]
1853,Title: sooo good; Content: tastes so good. Worth the money. My boyfriend hates wheat pasta and LOVES this. cooks fast tastes great.I love this brand and started buying more of their pastas. Bulk is best.,[0.4894696697158212]
523,"Title: not gnocchi; Content: The package says that this pasta is gnocchi, but it's actually just whole wheat pasta in the shape of shells.",[0.471349890631323]


In [38]:
search_reviews(df, "bad delivery", n=3)

,combined,similarity
1634,"Title: great product, poor delivery; Content: The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backe...",[0.49721102391966837]
1766,"Title: great product, poor delivery; Content: The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backe...",[0.49721102391966837]
1874,"Title: great product, poor delivery; Content: The coffee is excellent and I am a repeat buyer. Problem this time was with the UPS delivery. They left the box in front of my garage door in the middle of the driveway. Because of this odd delivery location, my wife ran over the box when she backe...",[0.49721102391966837]


In [39]:
search_reviews(df, "spoilt", n=1)

,combined,similarity
1474,Title: Disappointed; Content: The metal cover has severely disformed. And most of the cookies inside have been crushed into small pieces. Shopping experience is awful. I'll never buy it online again.,[0.3507283342740067]


In [40]:
search_reviews(df, "Rodeo Drive", n=2)

,combined,similarity
1660,Title: Rodeo Drive is Crazy Good Coffee!; Content: Rodeo Drive is my absolute favorite and I'm ready to order more! That's if I can find it.<br />I don't know why they are discontinuing it.<br />It arrived very fast.,[0.5124681511015045]
1700,Title: Rodeo Drive is Crazy Good Coffee!; Content: Rodeo Drive is my absolute favorite and I'm ready to order more! That's if I can find it.<br />I don't know why they are discontinuing it.<br />It arrived very fast.,[0.5124612663120206]


In [41]:
search_reviews(df, "Will switching to decaf tea at night helpful?", n=3)

,combined,similarity
1707,Title: breakfast tea; Content: We switch to this decaf tea at night for a great cup of tea and no sleep problems. Thanks for a good cup of tea.,[0.6277919493772717]
1824,Title: breakfast tea; Content: We switch to this decaf tea at night for a great cup of tea and no sleep problems. Thanks for a good cup of tea.,[0.6277751520250008]
1451,Title: breakfast tea; Content: We switch to this decaf tea at night for a great cup of tea and no sleep problems. Thanks for a good cup of tea.,[0.6277221963717288]


| **Method**                          | **Description**                                                                                           | **Advantages**                                | **Use Cases**                                                    |
|-------------------------------------|-----------------------------------------------------------------------------------------------------------|------------------------------------------------|------------------------------------------------------------------|
| **Cosine Similarity**                | Measures the cosine of the angle between two vectors.                                                     | Simple, effective, and widely used.            | General-purpose similarity search in embeddings.                 |
| **Dot Product (Inner Product)**      | Measures the direct product of two vectors.                                                               | Computationally efficient.                     | Tasks where embeddings are normalized.                           |
| **Euclidean Distance**               | Measures the straight-line distance between two vectors in space.                                         | Considers magnitude, simple to compute.        | Use cases where magnitude and scale matter.                      |
| **BM25 (Okapi BM25)**                | A ranking function based on term frequency and document length normalization.                             | Effective in traditional text retrieval.       | Initial retrieval in hybrid search systems.                      |
| **Approximate Nearest Neighbor (ANN)** | Techniques like LSH and HNSW for finding approximate nearest neighbors in large datasets.                 | Scales well to large datasets, fast retrieval. | Large-scale retrieval with trade-offs in accuracy.               |
| **Dual Encoder Models**             | Uses separate encoders for queries and documents and calculates similarity via learned functions.         | Effective for embedding-based retrieval.       | Initial retrieval or ranking tasks where embedding similarity is key. |
| **Cross-Encoder Models**             | Uses both query and document together to output a relevance score directly.                               | High accuracy in relevance scoring.            | Re-ranking top results for improved relevance.                   |
| **Attention Mechanisms**             | Dynamically weighs parts of the query and document embeddings to refine similarity scores.               | Handles complex retrieval tasks effectively.   | Multi-step reasoning and complex retrieval processes.            |


#### STOP HERE

#### Implement BM25

In [ ]:
#pip install rank-bm25

In [38]:
from rank_bm25 import BM25Okapi
import numpy as np

In [39]:
# Function to preprocess and tokenize text
def tokenize(text):
    # Tokenization logic here, e.g., using simple whitespace split
    return text.lower().split()

In [40]:
# Prepare the corpus and query
def search_reviews_bm25(df, query, n=5):
    # Tokenize the corpus
    tokenized_corpus = [tokenize(doc) for doc in df['combined']]
    
    # Initialize BM25
    bm25 = BM25Okapi(tokenized_corpus)
    
    # Tokenize the query
    tokenized_query = tokenize(query)
    
    # Get BM25 scores
    scores = bm25.get_scores(tokenized_query)
    
    # Add scores to DataFrame
    df['bm25_score'] = scores
    
    # Sort by BM25 score and select top-n
    df_sorted = df.sort_values('bm25_score', ascending=False)
    
    # Selecting specific columns (e.g., 'combined', 'bm25_score') from the sorted DataFrame
    results_df = df_sorted[['combined', 'bm25_score']].head(n)
    
    return results_df

In [41]:
# Example usage
query = "delicious beans"

top_results = search_reviews_bm25(df, query)
top_results

,combined,bm25_score
1596,"Title: Best beans your money can buy; Content: These are, hands down, the best jelly beans on the market. There isn't a gross one in the bunch and each of them has an intense, delicious flavor. Though I hesitate to pick a favorite, I have to say that I love green apple, a rare flavor in drugst...",7.570373
1743,"Title: Panama green beans; Content: These beans have produced some enjoyable cups of coffee with a medium roast (dark brown beans with no oil on their surfaces) at 455F for 17 minutes. We have experimented, using the same roasted beans with different water sources. We have learned that the beans...",7.454137
968,"Title: Amazing; Content: Nothing makes me feel at home more than a pot of blue runner red beans on the stove!! These red beans are the best. I'm so glad I can buy them here on amazon, my first two years of living away from Homs I had to bring them back with me or have someone send some! Now I ha...",6.233504
1088,"Title: Delicious!; Content: I enjoy this white beans seasoning, it gives a rich flavor to the beans I just love it, my mother in law didn't know about this Zatarain's brand and now she is traying different seasoning and she likes it very much.<br />Thank you Amazon for having it because now I ca...",6.071359
489,Title: coffee beans; Content: Coffee beans did not seem fresh. No oil on them what so ever. I have tasted much better and fresher. Will not order again.,5.927993


#### Key Points:
- **Tokenization:** The `tokenize` function should preprocess the text by converting it to lowercase and splitting it into tokens. You might use more sophisticated tokenization depending on your requirements.
- **BM25 Initialization:** `BM25Okapi` is initialized with the tokenized corpus.
- **Scoring:** BM25 scores are computed for the query and added to the DataFrame.
- **Sorting:** The DataFrame is sorted based on BM25 scores to retrieve the top-n results.

#### Explanation:
- **Tokenization:** Converts text to a list of tokens. This is crucial for BM25 to work effectively.
- **BM25 Scores:** Calculated using the `get_scores` method, which returns a list of scores for each document in the corpus.
- **Sorting and Selection:** The DataFrame is sorted based on the BM25 scores, and the top results are selected.


#### Implement Cross-Encoder Models

In [ ]:
#pip install sentence-transformers

In [ ]:
from sentence_transformers import CrossEncoder

In [ ]:
# Load the pre-trained Cross-Encoder model
model_name = 'cross-encoder/ms-marco-TinyBERT-L-6'
model = CrossEncoder(model_name)

In [ ]:
# Function to compute scores for query-document pairs
def score_pairs(query, documents):
    # Create input pairs
    pairs = [(query, doc) for doc in documents]
    
    # Compute scores
    scores = model.predict(pairs)
    
    return scores

In [ ]:
# Function to search through the DataFrame using Cross-Encoder
def search_reviews_cross_encoder(df, query, n=5):
    # Extract documents
    documents = df['combined'].tolist()
    
    # Compute scores for query-document pairs
    scores = score_pairs(query, documents)
    
    # Add scores to DataFrame
    df['cross_encoder_score'] = scores
    
    # Sort by Cross-Encoder score and select top-n
    df_sorted = df.sort_values('cross_encoder_score', ascending=False)
    
    # Selecting specific columns (e.g., 'combined', 'cross_encoder_score') from the sorted DataFrame
    results_df = df_sorted[['combined', 'cross_encoder_score']].head(n)
    
    return results_df

In [ ]:
%%time
# Example usage
# can take more than an hour
query = "delicious beans"

top_results = search_reviews_cross_encoder(df, query)
top_results

#### Key Points:

- **Loading the Model:** 
  - `CrossEncoder` from `sentence-transformers` is used to load a pre-trained Cross-Encoder model. 
  - The `model_name` can be adjusted to any other suitable pre-trained Cross-Encoder model.

- **Scoring Function:** 
  - The `score_pairs` function generates query-document pairs and computes their relevance scores using the Cross-Encoder model.

- **Integration:** 
  - The `search_reviews_cross_encoder` function integrates this scoring mechanism into your search pipeline, sorting documents based on relevance scores.

#### Explanation:

- **Model:** 
  - The `CrossEncoder` model is used to jointly process query-document pairs and calculate relevance scores.

- **Pairing:** 
  - The `score_pairs` function pairs each document with the query and uses the model to compute a relevance score for each pair.

- **Sorting and Selection:** 
  - Scores are added to the DataFrame, which is then sorted based on these scores to retrieve the top results.

This approach provides a high-accuracy method for document retrieval by leveraging Cross-Encoder models to assess the relevance of documents with respect to a query.
